# AIR Demo — Alice (Keygen rounds)

Reconciles Alice's sifted **Z⊗Z key-gen** bits (from `data/data4eat-sep/`) with
Bob using the separate-IR AIR protocol: pure-shortening polar code at
N = 2^20, SCL = 8.

- Assumed QBER: 0.0365 (95% two-sided Wald upper bound on the ZZ calibration
  BER, ceiled to 5e-4)
- Three AIR stages: f = 1.05, f = 1.10, then a union-bound stage targeting
  99.999% success probability

Run **Bob's notebook first** (`qkd-air-demo-bob-keygen.ipynb`), then this one.

In [ ]:
import sys
import os
import time
import gc
from datetime import datetime
from copy import deepcopy

sys.path.insert(0, '../../submodules/py_aff3ct/build/lib')
import numpy as np
import py_aff3ct

sys.path.insert(0, '../../src/')
from utils import (
    generate_shortened_from_file, generate_punctured_from_file,
    generate_shortened_BGL, generate_punctured_BGL,
    generate_frozen_position_from_file,
    num_frozen_bits_from_union_bound_from_file,
    mask_fillor, extractor,
    NumpyArrayHttpSender, NumpyArrayHttpReceiver,
)
from py_aff3ct.module.encoder import Encoder_polar
from py_aff3ct.module.crc import CRC_polynomial


## Load experimental data

Reads the `data4eat-sep` `.npy` files. Alice needs only her own result and
the round classification published during key sifting (`exp_type == 1` = key-gen
round, `exp_type == 0` = Bell-test round). She does **not** use Bob's measurement
basis: in the protocol only Alice's basis array is public, and Bob publishes the
per-round classification after combining it with his own (private) basis.

In [ ]:
DATA_DIR = '../../data/data4eat-sep/'

alice_result_raw = np.load(os.path.join(DATA_DIR, 'alice_result.npy')).astype(np.int8)
alice_exp_type   = np.load(os.path.join(DATA_DIR, 'alice_exp_type.npy'))

# Keep only ZZ key-gen rounds. Alice classifies rounds from the sifting string
# that Bob publishes after receiving her (public) basis array -- `exp_type == 1`
# alone -- and never uses Bob's measurement basis. (Bob's basis is 2 for every
# key-gen round by the sifting rule, but Alice does not need to know that.)
zz_mask = (alice_exp_type == 1)
num_total = int(zz_mask.sum())
print(f'Total rounds: {len(alice_exp_type)}, ZZ key-gen rounds: {num_total}')


## Three-step bit-flip pipeline (Alice's side)

Alice only applies the **AIR flip** (step 3): `a → 1−a` for all positions.
Steps 1–2 (SNSPD correction, per-subgroup CHSH flip) are Bob's responsibility;
the CHSH flip is skipped anyway for Z⊗Z key-gen rounds.

In [ ]:
alice_result_zz = alice_result_raw[zz_mask]
result_ir = (1 - alice_result_zz).astype(np.int32)
len_sifted_key = result_ir.shape[0]
print(f'Sifted key length: {len_sifted_key}')


## Configure parameters

- Assumed QBER = 0.0365 — 95% **two-sided** Wald upper bound on the ZZ
  calibration BER (point estimate 0.033115, σ = 0.001684, upper 0.036415),
  ceiled to 5e-4. The polar-code reliability file is generated at this BER.
- Real QBER is computed from the locally available dataset — used only for
  reporting the achieved efficiency.
- SCL = 8, CRC = 32, pure shortening S = N − L, P = 0
  (configuration mirrors `air-mc-keygen-calibration.py`).

In [ ]:
# --- Assumed QBER -----------------------------------------------------
# 0.0365 = 95% two-sided Wald upper bound on the ZZ keygen calibration BER
# (point estimate 0.033115, sigma 0.001684, upper 0.036415), ceiled to 5e-4.
# The polar-code reliability file n20_sp-bsc_ber0.0365_mu40_*.pc exists at
# this exact BER.
assumed_q_ber = 0.0365

# Real QBER, computed from the (locally available) dataset after the 3-step
# flip pipeline -- used only for reporting the achieved efficiency.
# In a real deployment this would come from the calibration point estimate.
# (Mirrors ir-raw-keygen.py: Alice AIR flip + Bob SNSPD correction.)
alice_result = np.load(os.path.join(DATA_DIR, 'alice_result.npy')).astype(np.int8)
bob_result_raw = np.load(os.path.join(DATA_DIR, 'bob_result.npy')).astype(np.int8)
alice_basis = np.load(os.path.join(DATA_DIR, 'alice_basis.npy'))
snspd_click = np.load(os.path.join(DATA_DIR, 'snspd_click.npy'))
alice_ir = (1 - alice_result[zz_mask]).astype(np.int32)
bob_zz = bob_result_raw[zz_mask].copy()
_flip = (alice_basis[zz_mask] == 1) & (snspd_click[zz_mask] == 1)
bob_zz[_flip] = 1 - bob_zz[_flip]
q_ber_stat = float(np.logical_xor(alice_ir, bob_zz.astype(np.int32)).sum() / len_sifted_key)
print(f'Real BER (after 3-step flip): {q_ber_stat:.6f}')

shannon_H2 = lambda x: -x * np.log2(x) - (1 - x) * np.log2(1 - x) if 0 < x < 1 else 0.0

n = int(np.ceil(np.log2(len_sifted_key)))
block_encoded_bits = 2 ** n
scl_len = 8
crc_size = 32

# Pure shortening: S = N - L, P = 0 (configuration mirrors air-mc-keygen-calibration.py)
S_BITS = block_encoded_bits - len_sifted_key
P_BITS = 0
assert S_BITS == 187329, \
    f'S_BITS = {S_BITS} != 187329: the sp/pe reliability files were generated for S = 187329'

ideal_frozen_bits = len_sifted_key * shannon_H2(q_ber_stat)
assumed_ideal_frozen_bits = len_sifted_key * shannon_H2(assumed_q_ber)
crc_efficiency_gain = crc_size / assumed_ideal_frozen_bits

print(f'N={block_encoded_bits}, L={len_sifted_key}, S={S_BITS}, P={P_BITS}')
print(f'Assumed QBER={assumed_q_ber}, real QBER={q_ber_stat:.6f}')


## Shortened / punctured positions (pure shortening)

The keygen codec uses **pure shortening**: S = N − L = 187329 shortened bits,
P = 0 punctured bits. Positions are loaded from
`conf/polar/sp-bsc/shortened_positions_N=20_S=187329.txt` when available
(generated by polar-fbgen-rust), else recomputed via BGL.

In [ ]:
PC_DIR = '../../conf/polar/sp-bsc/'

sp_file_S = os.path.join(PC_DIR, f'shortened_positions_N={n}_S={S_BITS}.txt')
sp_file_P = os.path.join(PC_DIR, f'punctured_positions_N={n}_P={P_BITS}.txt')

if os.path.exists(sp_file_S):
    shortened_idx = generate_shortened_from_file(sp_file_S)
    print(f'Shortened positions loaded from file: {len(shortened_idx)} positions')
else:
    shortened_idx = generate_shortened_BGL(block_encoded_bits, S_BITS)
    print(f'Shortened positions generated via BGL: {len(shortened_idx)} positions')

if os.path.exists(sp_file_P):
    punctured_idx = generate_punctured_from_file(sp_file_P)
    print(f'Punctured positions loaded from file: {len(punctured_idx)} positions')
else:
    punctured_idx = generate_punctured_BGL(block_encoded_bits, P_BITS)
    print(f'Punctured positions generated via BGL: {len(punctured_idx)} positions')

# Sanity: disjoint
overlap = set(shortened_idx.tolist()) & set(punctured_idx.tolist())
assert len(overlap) == 0, f'Shortened and punctured sets overlap at {len(overlap)} positions'

shortened_vec = np.zeros(block_encoded_bits, dtype=bool)
shortened_vec[shortened_idx] = True
punctured_vec = np.zeros(block_encoded_bits, dtype=bool)
punctured_vec[punctured_idx] = True
sp_vec = shortened_vec | punctured_vec
sp_idx = np.concatenate([shortened_idx, punctured_idx])

non_shortened_mask = (~sp_vec).astype(np.int32).reshape((1, -1))
non_sp_1d = np.where(non_shortened_mask.flatten() == 1)[0]
assert len(non_sp_1d) == len_sifted_key, \
    f'non-sp count {len(non_sp_1d)} != L {len_sifted_key}'
print(f'  Transmitted positions: {len(non_sp_1d)} (== L)')


## AIR efficiency stages and frozen vectors

Three stages:
1. f = 1.05 (configured efficiency vs. assumed ideal)
2. f = 1.10
3. f from the union bound (`frozen_bits` PE file) targeting **99.999% success**
   probability (failure rate ε = 1e-5)

The CRC cost is subtracted before converting efficiencies to effective
frozen-bit counts, exactly as in `air-mc-keygen-calibration.py`.

In [ ]:
# --- AIR stages -------------------------------------------------------
# 1. f = 1.05 (configured efficiency vs. assumed ideal)
# 2. f = 1.10
# 3. f from the union bound on the frozen_bits PE file, targeting 99.999%
#    success probability (failure rate eps = 1e-5)
pe_file = os.path.join(
    PC_DIR,
    f'frozen_bits_mu=40_N={n}_p={assumed_q_ber}_shortened={S_BITS}_punctured={P_BITS}.txt',
)
assert os.path.exists(pe_file), f'PE file not found: {pe_file}'

kf_ub = num_frozen_bits_from_union_bound_from_file(pe_file, sp_idx, 1e-5)
eff_ub = kf_ub / assumed_ideal_frozen_bits
configured_efficiency_stage = np.array([1.05, 1.10, eff_ub], dtype=np.float64)

# Subtract the CRC cost before converting efficiencies to frozen-bit counts,
# exactly as in air-mc-keygen-calibration.py.
efficiency_stage = configured_efficiency_stage - crc_efficiency_gain
effective_frozen_bits_stage = np.asarray(efficiency_stage * assumed_ideal_frozen_bits, dtype=np.int64)
effective_efficiency_stage = effective_frozen_bits_stage / assumed_ideal_frozen_bits
info_bits_stage = block_encoded_bits - effective_frozen_bits_stage - S_BITS - P_BITS
assert (info_bits_stage > 0).all(), f'info_bits must be > 0, got {info_bits_stage}'
assert np.all(np.diff(effective_frozen_bits_stage) > 0), 'frozen bits must be strictly increasing'

print(f'Union-bound frozen bits @99.999% success: {kf_ub} (eff = {eff_ub:.6f})')
print(f'Configured efficiency stages: {configured_efficiency_stage.tolist()}')
print(f'Effective frozen bits       : {effective_frozen_bits_stage.tolist()}')
print(f'Info bits                   : {info_bits_stage.tolist()}')

# --- Frozen vectors per stage -----------------------------------------
fb_file_dir = os.path.join(
    PC_DIR,
    f'n{n}_sp-bsc_ber{assumed_q_ber}_mu40_shortened{S_BITS}_punctured{P_BITS}.pc',
)
assert os.path.exists(fb_file_dir), f'.pc file not found: {fb_file_dir}'
print(f'Loading polar code: {fb_file_dir}')

frozen_vec_stage = []
effective_frozen_vec_stage = []
for efb in effective_frozen_bits_stage:
    _, _, eff_frozen_idx = generate_frozen_position_from_file(
        fb_file_dir, int(efb), sp_idx, verbose=False)
    fv = sp_vec.copy()
    fv[eff_frozen_idx] = True
    ev = np.zeros(block_encoded_bits, dtype=bool)
    ev[eff_frozen_idx] = True
    frozen_vec_stage.append(fv)
    effective_frozen_vec_stage.append(ev)

effective_frozen_vec_mask_stage = [ev.astype(np.int32).reshape((1, -1)) for ev in effective_frozen_vec_stage]
print(f'{len(frozen_vec_stage)} stages prepared')


## Build encoder and CRC

Alice encodes her padded key, verifies the BGL pure-shortening property
(codeword bits at shortened positions are 0), and computes the CRC tag of
the codeword.

In [ ]:
non_shortened_mask = (~sp_vec).astype(np.int32).reshape((1, -1))

enc = Encoder_polar(int(info_bits_stage[0]), block_encoded_bits, frozen_vec_stage[0])
enc.set_frozen_bits(frozen_vec_stage[0])
crc_alice = CRC_polynomial(block_encoded_bits, '0x04C11DB7', crc_size)
ext_crc = extractor(block_encoded_bits + crc_size, block_encoded_bits, block_encoded_bits + crc_size)
padder_alice = mask_fillor(len_sifted_key, block_encoded_bits, dtype=np.int32)

fixed_alice = result_ir.reshape((1, -1))
padded_alice = np.zeros((1, block_encoded_bits), dtype=np.int32)
padder_alice['fill ::output'].bind(padded_alice)
padder_alice['fill ::mask'].bind(non_shortened_mask)
padder_alice['fill ::input'].bind(fixed_alice)
padder_alice['fill'].exec()
print(f'Alice padded key shape: {padded_alice.shape}')

encoded_alice = np.zeros_like(padded_alice)
enc['light_encode ::U_N'].bind(padded_alice)
enc['light_encode ::X_N'].bind(encoded_alice)
enc['light_encode'].exec()

# BGL pure-shortening sanity: codeword bits at shortened positions must be 0
n_short_nonzero = int(encoded_alice.flatten()[shortened_idx].sum())
print(f'Shortened positions non-zero in encoded codeword: {n_short_nonzero} (must be 0)')
assert n_short_nonzero == 0

enc_crc_buf = np.zeros((1, block_encoded_bits + crc_size), dtype=np.int32)
crc_alice['build ::U_K1'].bind(encoded_alice)
crc_alice['build ::U_K2'].bind(enc_crc_buf)
crc_alice['build'].exec()

alice_crc_tag = np.zeros((1, crc_size), dtype=np.int32)
ext_crc['extract ::input'].bind(enc_crc_buf)
ext_crc['extract ::output'].bind(alice_crc_tag)
ext_crc['extract'].exec()
print(f'CRC tag computed, {crc_size} bits')


## Set up communication and logging

Alice sends on port 8003 and listens on port 8012 (Bob mirrors the roles).
Logs go to `log/eat-diqkd-sep/ir_demo_keygen/`.

In [ ]:
role = 'alice'
SENDER_PORT = 8003
RECEIVER_PORT = 8012
log_dir = '../../log/eat-diqkd-sep/ir_demo_keygen'
os.makedirs(log_dir, exist_ok=True)
start_time = datetime.now()
log_file = os.path.join(log_dir, f'qkd_air_demo_{role}_keygen_assumed_{assumed_q_ber}_{start_time.strftime("%Y%m%d_%H%M%S")}.txt')

class Logger:
    def __init__(self, log_file):
        self.log_file = log_file
        self.terminal = sys.stdout
    def write(self, message):
        self.terminal.write(message)
        with open(self.log_file, 'a') as f:
            f.write(message)
    def flush(self):
        self.terminal.flush()
        with open(self.log_file, 'a') as f:
            f.flush()

sys.stdout = Logger(log_file)
print(f'Log file: {log_file}')

sender_host = 'localhost'
sender_port = SENDER_PORT
receiver_host = 'localhost'
receiver_port = RECEIVER_PORT
array_sender = NumpyArrayHttpSender(host=sender_host, port=sender_port)
array_receiver = NumpyArrayHttpReceiver(host=receiver_host, port=receiver_port)
array_receiver.start_server()
print('HTTP communication ready')


## Run reconciliation

Alice sends the start signal, then the CRC tag, then — stage by stage — the
newly appended frozen bits of her encoded codeword, and waits for Bob's
CRC-check status. Zero status = success.

In [ ]:
num_bits_transmitted = 0
print('# ' * 16)
print(f'({datetime.now() - start_time}) Reconciliation Phase — Alice (Keygen)')
print('# ' * 16)

array_sender.send_array(np.ones(2))
print('Alice: Start the reconciliation phase')

time.sleep(5)

print(f'({datetime.now() - start_time}) Alice: Send CRC tag')
num_bits_transmitted += crc_size
array_sender.send_array(alice_crc_tag)
time.sleep(5)
print(f'Total {num_bits_transmitted} bits public')

failure_flag = True
for stage in range(len(frozen_vec_stage)):
    print(f'\n({datetime.now() - start_time}) Stage {stage+1}/{len(frozen_vec_stage)}')

    if stage == 0:
        indices = np.where(effective_frozen_vec_mask_stage[stage] == 1)
    else:
        indices = np.where(np.logical_xor(
            effective_frozen_vec_mask_stage[stage],
            effective_frozen_vec_mask_stage[stage-1]).astype(np.int32) == 1)
    num_appended = int(len(indices[1]))
    appended_bits = encoded_alice[indices].reshape(1, -1).astype(np.int32)

    print(f'Alice: Send {num_appended} appended frozen bits')
    num_bits_transmitted += num_appended
    array_sender.send_array(appended_bits)

    http_start = time.time()
    while array_receiver.received_array is None and (time.time() - http_start) < 1200:
        time.sleep(0.1)
    if array_receiver.received_array is not None:
        dec_status = array_receiver.received_array.copy()
        array_receiver.received_array = None
    else:
        print('Alice: Timeout waiting for dec_status')
        dec_status = None

    num_bits_transmitted += 1
    print(f'CRC check result: {dec_status}')

    if dec_status is not None and np.abs(dec_status).sum() == 0:
        failure_flag = False
        print(f'SUCCESS at stage {stage+1}')
        print(f'Assumed efficiency: {num_bits_transmitted/assumed_ideal_frozen_bits:.6f}')
        print(f'Real efficiency:    {num_bits_transmitted/ideal_frozen_bits:.6f}')
        break
    else:
        print(f'FAILED at stage {stage+1}')

if failure_flag:
    print(f'\n({datetime.now() - start_time}) All stages failed')


## Save Alice's sifted key

The AIR protocol never modifies Alice's data — she keeps her own
sifted key as-is. Save it for downstream use (only effective
positions: the sifted key is already L bits, no trimming applies).

In [ ]:
alice_out_file = os.path.splitext(log_file)[0] + '_alice.npy'
np.save(alice_out_file, result_ir)
print(f'Alice sifted key saved: {alice_out_file} ({result_ir.shape[0]} bits)')


## Cleanup

In [ ]:
del enc, crc_alice, ext_crc, padder_alice
del encoded_alice, padded_alice, enc_crc_buf, alice_crc_tag
del frozen_vec_stage, effective_frozen_vec_stage, effective_frozen_vec_mask_stage
gc.collect()

array_receiver.stop_server()
print(f'\nTotal elapsed: {datetime.now() - start_time}')
print(f'Log file: {log_file}')
